# VOID Model Cookbook

**Model:** netflix/void-model  
**Task:** Video-to-video inpainting (object and interaction deletion)  
**License:** Apache-2.0  
**Paper:** arXiv 2604.02296

VOID removes a target object and the interactions it caused in the scene, including secondary physical effects (for example, displaced or falling objects).

This cookbook follows the official model card and repository workflow and shows how to run Pass 1, optionally refine with Pass 2, and integrate VOID into production-style pipelines.

**Table of Contents:**
0. Setup and installation
1. Basic usage with official pipeline
2. Framework integration
3. Building agents around VOID
4. Advanced applications
5. Notes and references

## 0. Setup and Installation

The official workflow expects the VOID repository, base CogVideoX inpainting weights, and VOID checkpoints.

In [ ]:
# Core model and artifact constants used throughout this notebook.
MODEL = "netflix/void-model"
BASE_MODEL_REPO = "alibaba-pai/CogVideoX-Fun-V1.5-5b-InP"
PASS1_CHECKPOINT = "void_pass1.safetensors"
PASS2_CHECKPOINT = "void_pass2.safetensors"

DEFAULT_RESOLUTION = (384, 672)
MAX_FRAMES = 197
RECOMMENDED_GPU = "40GB+ VRAM (for example A100)"

print(f"MODEL: {MODEL}")
print(f"Base repo: {BASE_MODEL_REPO}")
print(f"Pass 1: {PASS1_CHECKPOINT}")
print(f"Pass 2: {PASS2_CHECKPOINT} (optional)")

In [ ]:
import subprocess
import sys

def run(cmd):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

# Clone the official repo (safe to skip if already cloned).
# run(["git", "clone", "https://github.com/netflix/void-model.git"])

# Install dependencies from the official repository.
# run([sys.executable, "-m", "pip", "install", "-r", "void-model/requirements.txt"])

# Optional for mask generation stage (documented by the project):
# 1) Set GEMINI_API_KEY
# 2) Install SAM2 separately
# The commands are environment-dependent and intentionally left commented.
# run(["git", "clone", "https://github.com/facebookresearch/sam2.git"])
# run([sys.executable, "-m", "pip", "install", "-e", "sam2"])

print("Setup cell loaded. Uncomment commands to execute in your environment.")

## 1. Basic Usage with Official Pipeline

VOID expects each sequence in a folder containing three files:

- input_video.mp4
- quadmask_0.mp4 (values: 0 remove, 63 overlap, 127 affected, 255 keep)
- prompt.json with a single bg key describing the post-removal background

In [ ]:
from pathlib import Path
import json

def validate_sequence_folder(seq_dir: str):
    seq_path = Path(seq_dir)
    required = ["input_video.mp4", "quadmask_0.mp4", "prompt.json"]
    missing = [name for name in required if not (seq_path / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing required files in {seq_dir}: {missing}")

    prompt_data = json.loads((seq_path / "prompt.json").read_text(encoding="utf-8"))
    if "bg" not in prompt_data:
        raise ValueError("prompt.json must include a 'bg' field")

    print("Sequence folder is valid:")
    print(f"- input: {(seq_path / 'input_video.mp4')}")
    print(f"- quadmask: {(seq_path / 'quadmask_0.mp4')}")
    print(f"- bg prompt: {prompt_data['bg']}")

# Example:
# validate_sequence_folder("void-model/sample/lime")

In [ ]:
import shlex

def build_pass1_command(
    repo_root="void-model",
    data_root="./sample",
    run_seq="lime",
    save_path="./outputs",
    transformer_path="./void_pass1.safetensors",
):
    cmd = [
        "python",
        "inference/cogvideox_fun/predict_v2v.py",
        "--config",
        "config/quadmask_cogvideox.py",
        f"--config.data.data_rootdir={data_root}",
        f"--config.experiment.run_seqs={run_seq}",
        f"--config.experiment.save_path={save_path}",
        f"--config.video_model.transformer_path={transformer_path}",
    ]
    rendered = " ".join(shlex.quote(part) for part in cmd)
    return f"cd {shlex.quote(repo_root)} && {rendered}"

pass1_cmd = build_pass1_command()
print(pass1_cmd)

# Environment-dependent execution:
# subprocess.run(pass1_cmd, shell=True, check=True)

## 2. Framework Integration

This section shows a light integration pattern for data engineering or app backends that orchestrate VOID runs as jobs.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class VoidJob:
    job_id: str
    sequence_name: str
    data_rootdir: str
    save_path: str
    pass1_checkpoint: str = "./void_pass1.safetensors"
    pass2_checkpoint: Optional[str] = None

def job_to_pass1_cmd(job: VoidJob, repo_root="void-model") -> str:
    return build_pass1_command(
        repo_root=repo_root,
        data_root=job.data_rootdir,
        run_seq=job.sequence_name,
        save_path=job.save_path,
        transformer_path=job.pass1_checkpoint,
    )

demo_job = VoidJob(
    job_id="job-001",
    sequence_name="lime",
    data_rootdir="./sample",
    save_path="./outputs",
)
print(job_to_pass1_cmd(demo_job))

## 3. Building Agents Around VOID

A practical agent pattern is to separate planning (validate inputs and prompts) from execution (run Pass 1 or Pass 1+Pass 2).

In [ ]:
import asyncio
from typing import Dict

async def plan_void_run(sequence_dir: str, bg_prompt: str) -> Dict[str, str]:
    seq = Path(sequence_dir)
    if not seq.exists():
        raise FileNotFoundError(f"Sequence directory not found: {sequence_dir}")
    if not bg_prompt.strip():
        raise ValueError("bg prompt cannot be empty")

    return {
        "sequence_dir": str(seq),
        "prompt": bg_prompt.strip(),
        "mode": "pass1",
    }

async def main():
    plan = await plan_void_run(
        sequence_dir="void-model/sample/lime",
        bg_prompt="A lime falls on the table.",
    )
    print(plan)

# await main()

## 4. Advanced Applications

Pass 2 is optional and improves temporal consistency on longer clips by using warped-noise refinement.

Use Pass 1 alone for most workflows, and add Pass 2 for quality-sensitive output where flicker consistency is critical.

In [ ]:
def build_batch_commands(sequence_names, data_root="./sample", save_root="./outputs"):
    cmds = []
    for seq in sequence_names:
        cmds.append(
            build_pass1_command(
                data_root=data_root,
                run_seq=seq,
                save_path=f"{save_root}/{seq}",
                transformer_path="./void_pass1.safetensors",
            )
        )
    return cmds

for c in build_batch_commands(["lime", "moving_ball", "pillow"]):
    print(c)

# For Pass 2 refinement, use the project-specific pass2 script and checkpoint path
# documented in the official VOID repository after generating Pass 1 outputs.

## 5. Notes and References

- Hugging Face model card: https://huggingface.co/netflix/void-model
- Source repository: https://github.com/netflix/void-model
- Colab notebook from authors: https://colab.research.google.com/github/netflix/void-model/blob/main/notebook.ipynb
- Paper: https://arxiv.org/abs/2604.02296

### Practical Constraints

- Recommended GPU is 40GB+ VRAM for official workflow.
- Default resolution is 384x672 and max frame count is 197.
- No hosted Inference Provider is listed on the model card at the time of writing.
- Keep environment-dependent execution commands commented until your runtime and paths are confirmed.